In [3]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/audio_lyrics"


Mounted at /content/drive


In [4]:
import os
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from pathlib import Path

In [5]:
import pandas as pd

DATASET_PATH = "/content/drive/MyDrive/audio_lyrics"
AUDIO_PATH  = "/content/drive/MyDrive/audio_lyrics/audio"
LYRICS_PATH = "/content/drive/MyDrive/audio_lyrics/lyrics"





In [6]:
#Audio Feature Extraction (MFCC)
def extract_mfcc(file_path, n_mfcc=40, max_len=130): # Added max_len parameter and default value
    y, sr = librosa.load(file_path, sr=22050)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)

    # Normalize each MFCC coefficient (zero mean, unit std)
    mfcc = (mfcc - mfcc.mean(axis=1, keepdims=True)) / (mfcc.std(axis=1, keepdims=True) + 1e-6)

    # Pad or truncate MFCCs to a fixed max_len
    if mfcc.shape[1] < max_len:
        pad_width = max_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), mode='constant')
    else:
        mfcc = mfcc[:, :max_len]

    return mfcc.astype(np.float32)

In [7]:
def extract_song_id(audio_filename):
    """
    en_0272_summertime.mp3 -> en_0272
    bn_0135_amar_sonar.mp3 -> bn_0135
    """
    base = os.path.splitext(audio_filename)[0]
    parts = base.split("_")
    return "_".join(parts[:2])


In [8]:
# Build lyrics index
lyrics_index = {}
for root, _, files in os.walk(LYRICS_PATH):
    for f in files:
        if f.lower().endswith(".txt"):
            key = os.path.splitext(f)[0].strip().lower()
            lyrics_index[key] = os.path.join(root, f)

def load_lyrics_embedding(song_id):
    key = song_id.strip().lower()
    if key not in lyrics_index:
        return np.zeros(TEXT_EMB_DIM, dtype=np.float32)

    with open(lyrics_index[key], "r", encoding="utf-8") as f:
        text = f.read().strip()

    if len(text) == 0:
        return np.zeros(TEXT_EMB_DIM, dtype=np.float32)

    return text_model.encode(text)


In [9]:
#Lyrics Embeddings (Text → Vector)
TEXT_EMB_DIM = 384
text_model = SentenceTransformer("all-MiniLM-L6-v2")

def load_lyrics_embedding(song_id):
    key = song_id.strip().lower()

    if key not in lyrics_index:
        # This should be rare now
        return np.zeros(TEXT_EMB_DIM, dtype=np.float32)

    with open(lyrics_index[key], "r", encoding="utf-8") as f:
        text = f.read().strip()

    if len(text) == 0:
        return np.zeros(TEXT_EMB_DIM, dtype=np.float32)

    return text_model.encode(text)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
class MusicDataset(Dataset):
    def __init__(self, audio_path):
        self.audio_files = []
        for root, _, files in os.walk(audio_path):
            for f in files:
                if f.lower().endswith(('.mp3', '.wav')):
                    self.audio_files.append(os.path.join(root, f))

        assert len(self.audio_files) > 0, f"No audio files found in {audio_path}"
        print(f"Found {len(self.audio_files)} audio files.")

    def __len__(self):
        return len(self.audio_files)

    def __getitem__(self, idx):
        file_path = self.audio_files[idx]
        file_name = os.path.basename(file_path)

        song_id = extract_song_id(file_name)
        mfcc = extract_mfcc(file_path)
        lyric_emb = load_lyrics_embedding(song_id)

        # Return tensors
        return (
            torch.tensor(mfcc, dtype=torch.float32).unsqueeze(0),  # (1, n_mfcc, T)
            torch.tensor(lyric_emb, dtype=torch.float32)
        )


In [11]:
dataset = MusicDataset(AUDIO_PATH)
print("Dataset length:", len(dataset))

loader = DataLoader(dataset, batch_size=16, shuffle=True)


Found 503 audio files.
Dataset length: 503


In [12]:
#encoder
class ConvEncoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(64*10*33, latent_dim)
        self.fc_logvar = nn.Linear(64*10*33, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc_mu(x), self.fc_logvar(x)


In [13]:
#decoder
class ConvDecoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 64*10*33)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 64, 10, 33)
        return self.deconv(x)


In [14]:
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU()
        )
        self.flatten = nn.Flatten()
        self.fc_mu = nn.Linear(64*5*17, latent_dim)  # adjust dynamically later if needed
        self.fc_logvar = nn.Linear(64*5*17, latent_dim)

        # Decoder
        self.fc_dec = nn.Linear(latent_dim, 64*5*17)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        B = x.size(0)
        enc = self.encoder(x)
        enc_flat = self.flatten(enc)
        mu = self.fc_mu(enc_flat)
        logvar = self.fc_logvar(enc_flat)
        z = self.reparameterize(mu, logvar)

        dec_flat = self.fc_dec(z)
        dec = dec_flat.view(B, 64, 5, 17)
        recon = self.decoder(dec)

        # 🔧 crop to input size (height x width)
        recon = recon[:, :, :x.shape[2], :x.shape[3]]

        return recon, mu, logvar


In [15]:

def vae_loss(recon, x, mu, logvar):
    mse = nn.functional.mse_loss(recon, x)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return mse + kl


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vae = ConvVAE(latent_dim=32).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)



In [17]:
mfcc, _ = next(iter(loader))
mfcc = mfcc.to(device)
recon, _, _ = vae(mfcc)

print("Input shape :", mfcc.shape)
print("Recon shape :", recon.shape)


Input shape : torch.Size([16, 1, 40, 130])
Recon shape : torch.Size([16, 1, 40, 130])


In [18]:
dataset = MusicDataset(AUDIO_PATH)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

vae = ConvVAE(latent_dim=32).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-4)  # smaller lr

for epoch in range(20):
    total_loss = 0
    for mfcc, _ in loader:
        mfcc = mfcc.to(device)

        recon, mu, logvar = vae(mfcc)
        loss = vae_loss(recon, mfcc, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")


Found 503 audio files.
Epoch 1, Loss: 1.2474
Epoch 2, Loss: 1.2390
Epoch 3, Loss: 1.2363
Epoch 4, Loss: 1.2375
Epoch 5, Loss: 1.2334
Epoch 6, Loss: 1.2295
Epoch 7, Loss: 1.2303
Epoch 8, Loss: 1.2289
Epoch 9, Loss: 1.2290
Epoch 10, Loss: 1.2288
Epoch 11, Loss: 1.2220
Epoch 12, Loss: 1.2182
Epoch 13, Loss: 1.2215
Epoch 14, Loss: 1.2166
Epoch 15, Loss: 1.2108
Epoch 16, Loss: 1.2093
Epoch 17, Loss: 1.2042
Epoch 18, Loss: 1.2009
Epoch 19, Loss: 1.1922
Epoch 20, Loss: 1.1856


In [19]:
vae.eval()


ConvVAE(
  (encoder): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (5): ReLU()
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc_mu): Linear(in_features=5440, out_features=32, bias=True)
  (fc_logvar): Linear(in_features=5440, out_features=32, bias=True)
  (fc_dec): Linear(in_features=32, out_features=5440, bias=True)
  (decoder): Sequential(
    (0): ConvTranspose2d(64, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), output_padding=(1, 1))
    (1): ReLU()
    (2): ConvTranspose2d(32, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), output_padding=(1, 1))
    (3): ReLU()
    (4): ConvTranspose2d(16, 1, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), output_padding=(1, 1))
  )
)

In [21]:
import numpy as np

latent_vectors = []

with torch.no_grad():
    for mfcc, _ in loader:
        mfcc = mfcc.to(device)
        # Corrected: Call vae(mfcc) to get recon, mu, and logvar
        recon, mu, logvar = vae(mfcc)
        z = mu  # use mean for clustering
        latent_vectors.append(z.cpu().numpy())

latent_vectors = np.concatenate(latent_vectors, axis=0)

print("Latent shape:", latent_vectors.shape)

Latent shape: (503, 32)


In [29]:
baseline_features = []

for mfcc, _ in loader:
    mfcc_mean = mfcc.mean(dim=2).squeeze(1)  # average over n_mfcc and remove singleton dimension
    baseline_features.append(mfcc_mean.numpy())

baseline_features = np.concatenate(baseline_features, axis=0)
print("Baseline shape:", baseline_features.shape)

Baseline shape: (503, 130)


In [30]:
from sklearn.cluster import KMeans

k = 5  # or 2 if Bangla vs English

kmeans_vae = KMeans(n_clusters=k, random_state=42)
labels_vae = kmeans_vae.fit_predict(latent_vectors)

kmeans_base = KMeans(n_clusters=k, random_state=42)
labels_base = kmeans_base.fit_predict(baseline_features)

In [31]:
languages = []
for file_path in dataset.audio_files:
    file_name = os.path.basename(file_path)
    song_id = extract_song_id(file_name)
    language_code = song_id.split('_')[0] # Assuming song_id is like 'en_0001'
    languages.append(language_code)

print(f"Populated 'languages' list with {len(languages)} entries.")
print("Sample languages:", languages[:5])

Populated 'languages' list with 503 entries.
Sample languages: ['hi', 'hi', 'hi', 'hi', 'hi']


In [32]:
from sklearn.cluster import AgglomerativeClustering

agg_vae = AgglomerativeClustering(n_clusters=k)
labels_agg_vae = agg_vae.fit_predict(latent_vectors)

agg_base = AgglomerativeClustering(n_clusters=k)
labels_agg_base = agg_base.fit_predict(baseline_features)

In [33]:
from sklearn.cluster import DBSCAN

db_vae = DBSCAN(eps=1.5, min_samples=5)
labels_db_vae = db_vae.fit_predict(latent_vectors)

db_base = DBSCAN(eps=1.5, min_samples=5)
labels_db_base = db_base.fit_predict(baseline_features)

In [34]:
from sklearn.metrics import silhouette_score

print("Silhouette VAE KMeans:", silhouette_score(latent_vectors, labels_vae))
print("Silhouette Baseline KMeans:", silhouette_score(baseline_features, labels_base))

Silhouette VAE KMeans: 0.2607204
Silhouette Baseline KMeans: 0.17532727


In [35]:
from sklearn.metrics import davies_bouldin_score

print("DB Index VAE:", davies_bouldin_score(latent_vectors, labels_vae))
print("DB Index Baseline:", davies_bouldin_score(baseline_features, labels_base))

DB Index VAE: 1.2454744603700938
DB Index Baseline: 1.6040081405846653


In [36]:
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import LabelEncoder

# Convert language labels to numerical labels for Adjusted Rand Index calculation
label_encoder = LabelEncoder()
true_labels = label_encoder.fit_transform(languages)

print("ARI VAE:", adjusted_rand_score(true_labels, labels_vae))
print("ARI Baseline:", adjusted_rand_score(true_labels, labels_base))

ARI VAE: -0.0013589247711021403
ARI Baseline: -0.002407673489783957
